In [ ]:
# Manual pipeline: raw config extraction -> template generation -> user fill-in
#
# This notebook demonstrates the manual calibration workflow:
# 1. Read raw file configurations
# 2. Identify unique channel configurations based on matching parameters
# 3. Generate calibration template files with null values for user to fill in
# 4. Generate mapping file (derived directly from raw data, no algorithm needed)
# 5. User fills in calibration values
# 6. Load filled templates and validate

from pathlib import Path
import yaml

# Raw file reading
from aa_si_calibration.raw_reader_api import (
    process_raw_folder,
    save_yaml,
    extract_unique_channels,
)

# Calibration template generation & validation
from aa_si_calibration.standardized_file_lib import (
    generate_calibration_templates,
    validate_loaded_templates,
    save_multi_channel_config_with_comments,
    build_short_filename_map,
)

# Mapping
from aa_si_calibration.mapping_algorithm import (
    build_mapping_from_raw_configs,
    get_calibration,
)

print("All imports successful!")

All imports successful!


In [ ]:
# Configuration - define input and output paths

# Record author - the individual running this pipeline and generating the records
RECORD_AUTHOR = "Placeholder Author"

# Filename style for single-channel calibration template files:
#   False -> long filenames derived from the full calibration key
#   True  -> short filenames: <date>_<freq_hz>_config-<N>.yml
SHORT_FILENAMES = True

# User-provided calibration date (required for template generation)
CALIBRATION_YEAR = 2026
CALIBRATION_MONTH = 2
CALIBRATION_DAY = 18

CALIBRATION_DATE = f"{CALIBRATION_YEAR:04d}-{CALIBRATION_MONTH:02d}-{CALIBRATION_DAY:02d}"

# Input folder
# RAW_INPUT_FOLDER = Path("./example_data/ek60_raw_file_input_folder")
RAW_INPUT_FOLDER = Path("./example_data/HB2407_raw")

# Output folder with subfolders for organization
OUTPUT_BASE = Path("./HB2407_Manual_Outputs")

# Create output subdirectories
RAW_CONFIGS_OUTPUT = OUTPUT_BASE / "raw_file_configs"
MAPPING_OUTPUT = OUTPUT_BASE / "mapping_files"
SINGLE_CAL_OUTPUT = OUTPUT_BASE / "Single_Channel_Calibration_Templates"

# Create all output directories
for folder in [RAW_CONFIGS_OUTPUT, MAPPING_OUTPUT, SINGLE_CAL_OUTPUT]:
    folder.mkdir(parents=True, exist_ok=True)

# Output file paths
RAW_CONFIGS_PATH = RAW_CONFIGS_OUTPUT / "raw_file_configs.yaml"
MAPPING_PATH = MAPPING_OUTPUT / "channel_to_calibration_mapping.yaml"
CALIBRATION_DICT_PATH = MAPPING_OUTPUT / "calibration_configurations.yaml"

print(f"Record author: {RECORD_AUTHOR}")
print(f"Short filenames: {SHORT_FILENAMES}")
print(f"\nInput folders:")
print(f"  Raw files: {RAW_INPUT_FOLDER}")
print(f"\nOutput folders:")
print(f"  Raw configs:              {RAW_CONFIGS_OUTPUT}")
print(f"  Mapping files:            {MAPPING_OUTPUT}")
print(f"  Calibration templates:    {SINGLE_CAL_OUTPUT}")
print(f"\nCalibration date: {CALIBRATION_DATE}")

Record author: Placeholder Author
Short filenames: True

Input folders:
  Raw files: example_data\HB2407_raw

Output folders:
  Raw configs:              HB2407_Manual_Outputs\raw_file_configs
  Mapping files:            HB2407_Manual_Outputs\mapping_files
  Calibration templates:    HB2407_Manual_Outputs\Single_Channel_Calibration_Templates

Calibration date: 2026-02-18


In [ ]:
# Step 1: Read raw file configurations
# Process all .raw files in the input folder and extract channel configurations.

file_configs, frequencies_set = process_raw_folder(RAW_INPUT_FOLDER, verbose=True)

# Save raw file configurations to YAML
save_yaml(file_configs, RAW_CONFIGS_PATH)
print(f"\nSaved raw file configurations to: {RAW_CONFIGS_PATH}")

Found 5 raw files in example_data\HB2407_raw
  - D20241112-T035837.raw
  - D20241112-T052842.raw
  - D20241112-T055441.raw
  - D20241112-T061349.raw
  - D20241112-T064333.raw
File: D20241112-T035837.raw
Instrument (detected): EK80
File format (from reader): EK80

--- GPS Summary ---
  NMEA datagrams found: 85195
  Valid GPS fixes: 3277
  First GPS: 41.520534, -71.344097
File: D20241112-T052842.raw
Instrument (detected): EK80
File format (from reader): EK80

--- GPS Summary ---
  NMEA datagrams found: 40036
  Valid GPS fixes: 1540
  First GPS: 41.520675, -71.344127
File: D20241112-T055441.raw
Instrument (detected): EK80
File format (from reader): EK80

--- GPS Summary ---
  NMEA datagrams found: 29317
  Valid GPS fixes: 1128
  First GPS: 41.520653, -71.344102
File: D20241112-T061349.raw
Instrument (detected): EK80
File format (from reader): EK80

--- GPS Summary ---
  NMEA datagrams found: 45745
  Valid GPS fixes: 1760
  First GPS: 41.520657, -71.344110
File: D20241112-T064333.raw
Instr

In [ ]:
# Step 2: Identify unique channel configurations
# The matching parameters are:
# - transceiver_id, transducer_model, transducer_serial_number, pulse_form,
# - frequency_start, frequency_end, transmit_power, transmit_duration_nominal
#
# Each unique combination of these parameters gets its own calibration template.
# Multiple raw files/channels sharing the same configuration will use the same
# calibration.

unique_channels = extract_unique_channels(file_configs, CALIBRATION_DATE)

print(f"Found {len(unique_channels)} unique channel configurations:")
for key, channel in unique_channels.items():
    print(f"\nChannel: {channel.get('channel_id')}")
    print(f"  Key: {key}")
    print(f"  Transceiver ID: {channel.get('transceiver_id')}")
    print(f"  Transducer model: {channel.get('transducer_model')}")
    print(f"  Transducer serial number: {channel.get('transducer_serial_number', 'N/A')}")
    print(f"  Pulse form: {channel.get('pulse_form')}")
    print(f"  Frequency: {channel.get('frequency_start')} - {channel.get('frequency_end')} Hz")
    print(f"  Transmit power: {channel.get('transmit_power')} W")
    print(f"  Transmit duration: {channel.get('transmit_duration_nominal')} s")

Found 5 unique channel configurations:

Channel: WBT 400479-15 ES18_ES
  Key: 2026-02-18__WBT 400479-15 ES18_ES__NoSN__0__0.001024__1000.0__18000.0__18000.0
  Transceiver ID: 400479
  Transducer model: ES18
  Transducer serial number: None
  Pulse form: 0
  Frequency: 18000.0 - 18000.0 Hz
  Transmit power: 1000.0 W
  Transmit duration: 0.001024 s

Channel: WBT 400528-15 ES38-7_ES
  Key: 2026-02-18__WBT 400528-15 ES38-7_ES__447__0__0.001024__1000.0__38000.0__38000.0
  Transceiver ID: 400528
  Transducer model: ES38-7
  Transducer serial number: 447
  Pulse form: 0
  Frequency: 38000.0 - 38000.0 Hz
  Transmit power: 1000.0 W
  Transmit duration: 0.001024 s

Channel: WBT 400503-15 ES70-7C_ES
  Key: 2026-02-18__WBT 400503-15 ES70-7C_ES__NoSN__0__0.001024__750.0__70000.0__70000.0
  Transceiver ID: 400503
  Transducer model: ES70-7C
  Transducer serial number: None
  Pulse form: 0
  Frequency: 70000.0 - 70000.0 Hz
  Transmit power: 750.0 W
  Transmit duration: 0.001024 s

Channel: WBT 400517

In [ ]:
# Step 3: Generate calibration template files
# Create template files with null values for the user to fill in.
# Each unique channel configuration gets its own template file.

calibration_templates = generate_calibration_templates(
    unique_channels,
    calibration_date=CALIBRATION_DATE,
    record_author=RECORD_AUTHOR,
    output_dir=SINGLE_CAL_OUTPUT,
    short_filenames=SHORT_FILENAMES,
)


Short key -> calibration parameters:

  18000 Hz:
    2026-02-18__18000__config-1:
      Model: ES18, Serial: None, Pulse form: 0
      Power: 1000.0 W, Duration: 0.001024 s

  38000 Hz:
    2026-02-18__38000__config-1:
      Model: ES38-7, Serial: 447, Pulse form: 0
      Power: 1000.0 W, Duration: 0.001024 s

  70000 Hz:
    2026-02-18__70000__config-1:
      Model: ES70-7C, Serial: None, Pulse form: 0
      Power: 750.0 W, Duration: 0.001024 s

  120000 Hz:
    2026-02-18__120000__config-1:
      Model: ES120-7C, Serial: None, Pulse form: 0
      Power: 250.0 W, Duration: 0.001024 s

  200000 Hz:
    2026-02-18__200000__config-1:
      Model: ES200-7C, Serial: None, Pulse form: 0
      Power: 150.0 W, Duration: 0.001024 s

Generated 5 calibration template file(s)
  Output directory: HB2407_Manual_Outputs\Single_Channel_Calibration_Templates
  Filename style: short
  Record created: 2026-04-14T17:00:33.675154+00:00
  Record author: Placeholder Author

Template files created:
  - 202

In [ ]:
# Step 4: Generate mapping file
# Since the calibration templates are derived directly from raw file channels,
# the mapping is deterministic - no matching algorithm needed.

# Build the mapping file (uses long internal keys)
mapping_dict = build_mapping_from_raw_configs(file_configs, CALIBRATION_DATE)

# When using short filenames, remap the mapping values to short identifiers
if SHORT_FILENAMES:
    filename_map = build_short_filename_map(unique_channels, calibration_date=CALIBRATION_DATE)
    for fname, channels in mapping_dict.items():
        for channel_id in channels:
            long_key = channels[channel_id]
            channels[channel_id] = filename_map.get(long_key, long_key)

# Save mapping file
with open(MAPPING_PATH, 'w') as f:
    yaml.dump(mapping_dict, f, default_flow_style=False, sort_keys=False)

print(f"Saved mapping file to: {MAPPING_PATH}")
print("\nMapping dictionary preview:")
for filename, channels in mapping_dict.items():
    print(f"\n{filename}:")
    for channel_id, cal_key in channels.items():
        print(f"  {channel_id}")
        print(f"    -> {cal_key}")

Saved mapping file to: HB2407_Manual_Outputs\mapping_files\channel_to_calibration_mapping.yaml

Mapping dictionary preview:

D20241112-T035837.raw:
  WBT 400479-15 ES18_ES
    -> 2026-02-18__18000__config-1
  WBT 400528-15 ES38-7_ES
    -> 2026-02-18__38000__config-1
  WBT 400503-15 ES70-7C_ES
    -> 2026-02-18__70000__config-1
  WBT 400517-15 ES120-7C_ES
    -> 2026-02-18__120000__config-1
  WBT 400509-15 ES200-7C_ES
    -> 2026-02-18__200000__config-1

D20241112-T052842.raw:
  WBT 400479-15 ES18_ES
    -> 2026-02-18__18000__config-1
  WBT 400528-15 ES38-7_ES
    -> 2026-02-18__38000__config-1
  WBT 400503-15 ES70-7C_ES
    -> 2026-02-18__70000__config-1
  WBT 400517-15 ES120-7C_ES
    -> 2026-02-18__120000__config-1
  WBT 400509-15 ES200-7C_ES
    -> 2026-02-18__200000__config-1

D20241112-T055441.raw:
  WBT 400479-15 ES18_ES
    -> 2026-02-18__18000__config-1
  WBT 400528-15 ES38-7_ES
    -> 2026-02-18__38000__config-1
  WBT 400503-15 ES70-7C_ES
    -> 2026-02-18__70000__config-1
  

In [ ]:
# Step 4 (continued): Save calibration configurations file
# The calibration_configurations.yaml file is an alternative representation
# of the same data as the individual template files. Users can choose to
# edit either one - both represent the same data structure.

save_multi_channel_config_with_comments(calibration_templates, CALIBRATION_DICT_PATH)

print(f"Saved calibration configurations to: {CALIBRATION_DICT_PATH}")

Saved calibration configurations to: HB2407_Manual_Outputs\mapping_files\calibration_configurations.yaml


# STEP 5: USER ACTION REQUIRED — Fill In Calibration Values

## STOP HERE AND FILL IN YOUR CALIBRATION DATA

The template files have been generated. Before proceeding to Step 6, you must fill in your calibration values.

### Option A: Edit Individual Template Files
Navigate to the output folder printed above (default: `<OUTPUT_BASE>/Single_Channel_Calibration_Templates/`).
Each file contains one channel's calibration template. Fill in the values marked as `null`.

**Note:** The calibration date (`calibration_date`) is already set from the configuration at the start of the notebook. You do not need to fill in this field.

### Option B: Edit the Consolidated File
Edit the single file at `<OUTPUT_BASE>/mapping_files/calibration_configurations.yaml`.
This contains all calibration templates in one file.

---

## Required Fields to Fill In:
| Field | Description | Example |
|-------|-------------|---------|
| `gain_correction` | Transducer gain [dB] | `[24.06]` |
| `sa_correction` | Sa correction [dB] | `[-0.68]` |
| `equivalent_beam_angle` | Equivalent beam angle [dB] | `-20.6` |
| `absorption_indicative` | Absorption coefficient [dB/m] | `0.0072` |
| `sound_speed_indicative` | Sound speed [m/s] | `1522.6` |

## Optional Fields:
- `beamwidth_transmit/receive_major/minor` — Beam angles
- `echoangle_major/minor` — Angle offsets
- `echoangle_major/minor_sensitivity` — Angle sensitivities
- `sample_interval`, `transmit_bandwidth`, etc.
- `calibration_comments`, `sphere_diameter`, `sphere_material`

---

**When you have filled in the calibration values, continue to Step 6 below.**

In [ ]:
# Step 6: Load filled calibration templates and validate
# After the user has filled in calibration values, load the templates
# and check for completeness.

loaded_templates, all_complete = validate_loaded_templates(SINGLE_CAL_OUTPUT)

Loaded 5 calibration template(s)

   Missing required fields: gain_correction, sa_correction, equivalent_beam_angle, absorption_indicative, sound_speed_indicative

   Missing required fields: gain_correction, sa_correction, equivalent_beam_angle, absorption_indicative, sound_speed_indicative

   Missing required fields: gain_correction, sa_correction, equivalent_beam_angle, absorption_indicative, sound_speed_indicative

   Missing required fields: gain_correction, sa_correction, equivalent_beam_angle, absorption_indicative, sound_speed_indicative

   Missing required fields: gain_correction, sa_correction, equivalent_beam_angle, absorption_indicative, sound_speed_indicative

Fill in the missing values before using the calibration data.


In [ ]:
# Step 6 (continued): Update calibration configurations from templates
# Regenerate the calibration_configurations.yaml file from the individual
# template files to ensure both representations are in sync.

calibration_dict = loaded_templates

save_multi_channel_config_with_comments(calibration_dict, CALIBRATION_DICT_PATH)

print(f"Updated calibration configurations: {CALIBRATION_DICT_PATH}")
print(f"  (Synced from {len(loaded_templates)} individual template files)")

Updated calibration configurations: HB2407_Manual_Outputs\mapping_files\calibration_configurations.yaml
  (Synced from 5 individual template files)


In [ ]:
# Step 7: Retrieve calibration for a specific raw file + channel

print("Example: retrieving calibration data")
print("-" * 40)

# Get the first file and first channel as an example
if mapping_dict:
    example_filename = list(mapping_dict.keys())[0]
    if mapping_dict[example_filename]:
        example_channel = list(mapping_dict[example_filename].keys())[0]
        
        # Retrieve calibration using the in-memory dictionaries
        cal_data = get_calibration(example_filename, example_channel, mapping_dict, calibration_dict)
        
        if cal_data:
            print(f"\nCalibration for '{example_filename}' -> '{example_channel}':")
            print(f"  Calibration date: {cal_data.get('calibration_date')}")
            print(f"  Frequency: {cal_data.get('frequency')}")
            print(f"  Gain correction: {cal_data.get('gain_correction')}")
            print(f"  Sa correction: {cal_data.get('sa_correction')}")
            print(f"  Equivalent beam angle: {cal_data.get('equivalent_beam_angle')}")
            print(f"  Sound speed: {cal_data.get('sound_speed_indicative')}")
            print(f"  Absorption: {cal_data.get('absorption_indicative')}")
        else:
            print(f"No calibration found for {example_filename} -> {example_channel}")

Example: retrieving calibration data
----------------------------------------

Calibration for 'D20241112-T035837.raw' -> 'WBT 400479-15 ES18_ES':
  Calibration date: 2026-02-18
  Frequency: [18000.0]
  Gain correction: [None]
  Sa correction: [None]
  Equivalent beam angle: None
  Sound speed: None
  Absorption: None


In [ ]:
# Pipeline summary

print("Manual pipeline complete - output files summary")
print("-" * 40)

def list_files_recursive(folder, indent=0):
    """List all files in a folder recursively."""
    if not folder.exists():
        return
    for item in sorted(folder.iterdir()):
        if item.is_dir():
            print("  " * indent + f"  {item.name}/")
            list_files_recursive(item, indent + 1)
        else:
            size_kb = item.stat().st_size / 1024
            print("  " * indent + f"  {item.name} ({size_kb:.1f} KB)")

print(f"\nOutput directory: {OUTPUT_BASE}")
print("-" * 40)
list_files_recursive(OUTPUT_BASE)

print("\nOutput file descriptions:")
print("""
1. raw_file_configs/raw_file_configs.yaml
   - Channel configurations extracted from each raw file
   - Useful for understanding what channels/frequencies are in your data

2. mapping_files/channel_to_calibration_mapping.yaml
   - Maps each raw file + channel to a calibration configuration key
   - Used to look up which calibration to use for each channel

3. mapping_files/calibration_configurations.yaml
   - Contains ALL calibration templates/data in a single file
   - Alternative to editing individual files (synced from templates)

4. Single_Channel_Calibration_Templates/*.yml
   - Individual template files for each unique channel configuration
   - Edit these to fill in your calibration values
""")

print("Workflow summary:")
print("""
Steps 1-4: Run automatically to extract raw file configs and generate templates
Step 5:    Fill in calibration values in template files
Step 6:    Run to validate and sync calibration data
Step 7:    Use the mapping to retrieve calibration for any file/channel

For echopype calibration, extract these parameters from the calibration data:
    - gain_correction
    - sa_correction  
    - equivalent_beam_angle
    - sound_speed_indicative
    - absorption_indicative
""")

Manual pipeline complete - output files summary
----------------------------------------

Output directory: HB2407_Manual_Outputs
----------------------------------------
  mapping_files/
    calibration_configurations.yaml (12.6 KB)
    channel_to_calibration_mapping.yaml (1.5 KB)
  raw_file_configs/
    raw_file_configs.yaml (18.2 KB)
  Single_Channel_Calibration_Templates/
    2026-02-18__120000__config-1.yaml (5.5 KB)
    2026-02-18__18000__config-1.yaml (5.5 KB)
    2026-02-18__200000__config-1.yaml (5.5 KB)
    2026-02-18__38000__config-1.yaml (5.5 KB)
    2026-02-18__70000__config-1.yaml (5.5 KB)

Output file descriptions:

1. raw_file_configs/raw_file_configs.yaml
   - Channel configurations extracted from each raw file
   - Useful for understanding what channels/frequencies are in your data

2. mapping_files/channel_to_calibration_mapping.yaml
   - Maps each raw file + channel to a calibration configuration key
   - Used to look up which calibration to use for each channel

3.